In [ ]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, uuid, lit

warehouse_path = "/tmp/ehdip_warehouse"
os.makedirs(warehouse_path, exist_ok=True)

spark = SparkSession.builder \
    .appName("Day02_Streaming") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.ehdip_bronze")
spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_bronze.raw_payloads (
    payload_id STRING,
    source_system_id STRING,
    raw_payload_json STRING,
    ingestion_timestamp TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

# Simulate streaming using a rate stream
mock_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

fhir_payload_json = '{"resourceType": "Observation", "status": "final", "code": {"text": "Heart Rate"}}'

processed_stream = mock_stream \
    .withColumn("payload_id", uuid()) \
    .withColumn("source_system_id", lit("simulated-stream")) \
    .withColumn("raw_payload_json", lit(fhir_payload_json)) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .select("payload_id", "source_system_id", "raw_payload_json", "ingestion_timestamp")

checkpoint_dir = "/tmp/ehdip_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

query = processed_stream.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_dir) \
    .toTable("local.ehdip_bronze.raw_payloads")

print("Started streaming query...")
time.sleep(10) # Let it run for 10 seconds to generate data
query.stop()
print("Stopped streaming query.")

spark.sql("SELECT * FROM local.ehdip_bronze.raw_payloads ORDER BY ingestion_timestamp DESC LIMIT 5").show(truncate=False)
